# 🎨 Paint-Code-RL: Zero-Cost GRPO Generative Art Training on Kaggle T4x2 GPU

This notebook trains a **Qwen2.5-Coder** policy via **GRPO (Group Relative Policy Optimization)** with Headless WebGL feedback and a **5-tier multi-signal visual reward matrix**:
- 🎯 **Semantic Prompt Alignment** (CLIP cosine similarity with art prompts)
- 🖼️ **Pixel-Space Visual Richness** (Anti-blank canvas & color palette entropy)
- 🖌️ **Natural Media Utilization** (`p5.brush` watercolor washes & textures)
- 🛡️ **Anti-Cheat Verifier** (Detects and penalizes text-in-canvas hacks)
- ⚡ **Execution Gate** (Headless Chromium WebGL execution)

In [ ]:
# 1. Install System Dependencies for Headless WebGL Chromium
!apt-get update -qq
!apt-get install -y -qq chromium-browser nodejs npm xvfb
!node -v && npm -v

In [ ]:
# 2. Install Python RL & Multimodal Dependencies
!pip install -q torch==2.5.1 transformers==4.49.0 trl==0.15.1 peft bitsandbytes accelerate pydantic safetensors datasets Pillow pyyaml requests

In [ ]:
# 3. Clone Repository and checkout latest branch
!git clone https://github.com/harshitthek/paint-code-rl.git
%cd paint-code-rl

In [ ]:
# 4. Install Renderer Node Modules
%cd renderer
!npm install --silent
%cd ..

In [ ]:
# 5. Start Headless WebGL Renderer in Background
import subprocess, time, requests

proc = subprocess.Popen(["node", "renderer/server.js"])
time.sleep(3)

# Verify health
res = requests.get("http://127.0.0.1:3000/health").json()
print("Renderer Health:", res)

In [ ]:
# 6. Run Hardware-Safe GRPO Training on Kaggle GPU
import os
os.environ["ENV"] = "kaggle"
os.environ["PYTHONUNBUFFERED"] = "1"

!python -u scripts/train_grpo.py --mode train --max-steps 200 --checkpoint-dir /kaggle/working/artifacts/checkpoints

In [ ]:
# 7. Generate Showcase Artworks with Trained Model
!python scripts/generate_and_render.py --output-dir /kaggle/working/artifacts/renders

# Display generated gallery in notebook
from IPython.display import display, HTML, Image
import glob

renders = glob.glob("/kaggle/working/artifacts/renders/*.png")
print(f"Generated {len(renders)} artworks:")
for r in renders[:5]:
    display(Image(filename=r, width=400))